# Data Challenge ENS QRT 2020

This notebook shows my approach to solve the challenge of data challenge ens QRT 2020.

## Used libraries

In [2]:
import seaborn as sns
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, SelectFromModel, SelectPercentile, f_classif

## Loading data

The train and test inputs are composed of 46 features.

The target of this challenge is `RET` and corresponds to the fact that the **return is in the top 50% of highest stock returns**.

Since the median is very close to 0, this information should not change much with the idea to predict the sign of the return.

In [6]:
train = pd.read_csv('../train_extended.csv', index_col='ID') # previously train_extended.csv
test = pd.read_csv('../test_extended.csv', index_col='ID') # previously test_extended.csv
x_train, y_train = train.drop('RET', axis=1), train['RET']

FileNotFoundError: [Errno 2] No such file or directory: '../train_extended.csv'

In [ ]:
# convert categorical columns to category type
categorical_features = ['STOCK','INDUSTRY','INDUSTRY_GROUP','SECTOR','SUB_INDUSTRY']
for col in categorical_features:
    x_train[col] = x_train[col].astype('category')
    test[col] = test[col].astype('category')


### Defining our ML Pipeline and the GridSearchCV procedure
- Throug a GridSearchCV we aim to compare the performance of different Imputations methods and Feature Selections
- For that we create a pipeline to embbed all the successive steps of the data transformation in one estimator
- Warning to use Catboost in a pipeline : 
    - https://medium.com/analytics-vidhya/combining-scikit-learn-pipelines-with-catboost-and-dask-part-2-9240242966a7
    - https://stackoverflow.com/questions/56742441/



In [ ]:

model = Pipeline([
    ('feature_selection', 'passthrough'),
    ('classify', 'passthrough')
])

# Define the parameter grid for GridSearchCV
param_grid = [
    {
        'feature_selection': [SelectFromModel(RandomForestClassifier(n_estimators=100, max_depth=8, n_jobs=-1))],
        'feature_selection__max_features': [75],
        'classify': [RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1), 
                    CatBoostClassifier(depth=6, iterations=1000,  early_stopping_rounds=100, task_type="GPU", devices='0', verbose=0)]
    },

    {
        'feature_selection': [SelectFromModel(CatBoostClassifier(depth=8, iterations=1000,  early_stopping_rounds=50, task_type="GPU", devices='0', verbose=0))],
        'feature_selection__max_features': [75],
        'classify': [RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1), 
                    CatBoostClassifier(depth=6, iterations=1000,  early_stopping_rounds=100, task_type="GPU", devices='0', verbose=0)]
    },
    
    {
        'feature_selection': [SelectKBest()],
        'feature_selection__score_func': [mutual_info_classif, f_classif],
        'feature_selection__k': [75],
        'classify': [RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1), 
                    CatBoostClassifier(depth=8, iterations=1000,  early_stopping_rounds=100, task_type="GPU", devices='0', verbose=0)],
    }
]

# define the GridSearchCV
grid = GridSearchCV(model, param_grid, cv=3, scoring='accuracy', verbose=3, n_jobs=1, error_score='raise')

# fit the GridSearchCV
grid.fit(x_train, y_train)

# get the parameters of models sorted by ranked score of accuracy
results = pd.DataFrame(grid.cv_results_)
results.sort_values(by='rank_test_score', inplace=True)
results

In [ ]:
imputer = SimpleImputer(strategy='mean').set_output(transform='pandas')
selector = SelectPercentile(f_classif, percentile=10).set_output(transform='pandas')
# clf = CatBoostClassifier(depth=6, iterations=1000,  early_stopping_rounds=100, verbose=0)
clf = RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1)

pipe = Pipeline([
    ('imputer', imputer),
    ('feature_selection', selector),
    ('classify', clf)
])

score = cross_val_score(pipe, x_train, y_train, cv=3, scoring='accuracy', n_jobs=-1, verbose=3)

print(score)


## Model and local score

A Random Forest (RF) model is chosen for the Benchmark. We consider a large number of tree with a quiet small depth. The missing values are simply filled with 0. A KFold is done on the dates (using `DATE`) for a local scoring of the model. 

**Ideas of improvements**: Tune the RF hyperparameters, deal with the missing values, change the features, consider another model, ...

In [ ]:
# Select the best feature by importing selectedFeaturesXGB.csv and selectedFeaturesRF.csv
selectedFeaturesXGB = pd.read_csv('../selectedFeaturesXGB.csv')
selectedFeaturesRF = pd.read_csv('../selectedFeaturesRF.csv')
# lets take the union of the two selected features
selectedFeaturesTrees = list(set(selectedFeaturesXGB['0']).union(set(selectedFeaturesRF['0'])))

# removing the features that are correlated than more than 0.95 with each other

# calculate the correlation matrix
corr_matrix = x_train[selectedFeaturesTrees].corr().abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# draw the heatmap
sns.heatmap(corr_matrix)
# Find index of feature columns with correlation greater than 0.8
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

to_drop


FileNotFoundError: [Errno 2] No such file or directory: '../selectedFeaturesXGB.csv'

In [ ]:
scores_f = pd.read_csv('../scores_f_classif.csv')
scores_mutual_info = pd.read_csv('../scores_mutual_info.csv')
# lets take the 50 best features from the scores_f and scores_mutual_info
selectedFeaturesF = list(scores_f['feature'][:100])
selectedFeaturesMI = list(scores_mutual_info['feature'][:100])

In [ ]:
train = pd.read_csv('../Benchmark_contestants/train_lucabri_cleaned_extended.csv', index_col='ID')
test = pd.read_csv('../Benchmark_contestants/test_lucabri_cleaned_extended.csv', index_col='ID')
y_train = train['RET']

In [14]:
fixed_features = test.columns

In [ ]:
ret_cols = [f'RET_{i}' for i in range(1, 6)]
vol_cols = [f'VOLUME_{i}' for i in range(1, 6)]
categorical_features = ['SECTOR', 'INDUSTRY_GROUP']
SECTOR_DATE_features = [ feature for feature in test.columns if 'SECTOR_DATE_' in feature]
WEEKLY_features = [ feature for feature in test.columns if 'WEEK' in feature]
technical_indicators_features = ['RET_10_day_momentum', 'VOLUME_10_day_momentum', '10_day_mean_RET_vola', '10_day_mean_VOLUME_vola', 'RSI_SECTOR_DATE_20', 'Sum_ADL']

In [31]:
# make the union of the selected features
features = list(set(ret_cols + vol_cols + categorical_features + SECTOR_DATE_features + WEEKLY_features + technical_indicators_features))
len(features)

46

In [ ]:
X_train = train[features]

# A quiet large number of trees with low depth to prevent overfits

train_dates = train['DATE'].unique()
test_dates = test['DATE'].unique()

n_splits = 4
scores = []
models = []
base_model = CatBoostClassifier(depth=6, iterations=1000,  early_stopping_rounds=100,  verbose=0, cat_features=[X_train.columns.get_loc('SECTOR'), X_train.columns.get_loc('INDUSTRY_GROUP')])

splits = KFold(n_splits=n_splits, random_state=0,
               shuffle=True).split(train_dates)

for i, (local_train_dates_ids, local_test_dates_ids) in enumerate(splits):
    local_train_dates = train_dates[local_train_dates_ids]
    local_test_dates = train_dates[local_test_dates_ids]

    local_train_ids = train['DATE'].isin(local_train_dates)
    local_test_ids = train['DATE'].isin(local_test_dates)

    X_local_train = X_train.loc[local_train_ids]
    y_local_train = y_train.loc[local_train_ids]
    X_local_test = X_train.loc[local_test_ids]
    y_local_test = y_train.loc[local_test_ids]

    X_local_train = X_local_train.fillna(0)
    X_local_test = X_local_test.fillna(0)

    model = base_model
    model.fit(X_local_train, y_local_train)
  
    y_local_pred = model.predict_proba(X_local_test)[:, 1]
    
    sub = train.loc[local_test_ids].copy()
    sub['pred'] = y_local_pred
    y_local_pred = sub.groupby('DATE')['pred'].transform(lambda x: x > x.median()).values

    models.append(model)
    score = accuracy_score(y_local_test, y_local_pred)
    scores.append(score)
    print(f"Fold {i+1} - Accuracy: {score* 100:.2f}%")

mean = np.mean(scores)*100
std = np.std(scores)*100
u = (mean + std)
l = (mean - std)
print(f'Accuracy: {mean:.2f}% [{l:.2f} ; {u:.2f}] (+- {std:.2f})')

In [33]:
print(f"Mean accuracy: {mean:.2f}%")
print(f"Std accuracy: {std:.2f}")
print(f'scores = {scores}')


Mean accuracy: 51.92%
Std accuracy: 0.71
scores = [0.5076350057631672, 0.52541381921348, 0.524424321073022, 0.5194464704606833]


In [ ]:
# cross validation score with predict built-in method
model = RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1)
score = cross_val_score(model, X_train, y_train, cv=4, scoring='accuracy', n_jobs=-1, verbose=3)

print("Cross-validation score with predict built-in method")
for i in range(len(score)):
    print(f'Fold {i+1} - Accuracy: {score[i]*100:.2f}%')
print(f'Accuracy: {np.mean(score)*100:.2f}% (+- {np.std(score)*100:.2f})')

In [ ]:
feature_importances = pd.DataFrame([model.feature_importances_ for model in models], columns=features)
# sns.set(rc={'figure.figsize':(10,16)})
sns.barplot(data=feature_importances, orient='h', order=feature_importances.mean().sort_values(ascending=False).index)

## Generate the submission

The same parameters of the RF model are considered. With that we build a new RF model on the entire `train` dataset. The predictions are saved in a `.csv` file.

In [34]:
target = 'RET'
X_test = test[features]

# model = RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1)
model = base_model
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]

sub = test.copy()
sub['pred'] = y_pred
y_pred = sub.groupby('DATE')['pred'].transform(
    lambda x: x > x.median()).values

submission = pd.Series(y_pred)
submission.index = test.index
submission.name = target

submission.to_csv('./submission_catboost_optimal_features_plus_sector_industry_group.csv', index=True, header=True)

Learning rate set to 0.135344
0:	learn: 0.6925685	total: 381ms	remaining: 6m 20s
1:	learn: 0.6921041	total: 750ms	remaining: 6m 14s
2:	learn: 0.6916930	total: 1.11s	remaining: 6m 8s
3:	learn: 0.6912927	total: 1.46s	remaining: 6m 2s
4:	learn: 0.6909287	total: 1.83s	remaining: 6m 3s
5:	learn: 0.6905166	total: 2.18s	remaining: 6m 1s
6:	learn: 0.6901797	total: 2.55s	remaining: 6m 1s
7:	learn: 0.6899512	total: 2.92s	remaining: 6m 1s
8:	learn: 0.6896034	total: 3.26s	remaining: 5m 59s
9:	learn: 0.6891941	total: 3.65s	remaining: 6m 1s
10:	learn: 0.6888690	total: 4.01s	remaining: 6m
11:	learn: 0.6885715	total: 4.37s	remaining: 5m 59s
12:	learn: 0.6882902	total: 4.83s	remaining: 6m 6s
13:	learn: 0.6880098	total: 5.35s	remaining: 6m 16s
14:	learn: 0.6877276	total: 5.73s	remaining: 6m 16s
15:	learn: 0.6875191	total: 6.08s	remaining: 6m 13s
16:	learn: 0.6871551	total: 6.47s	remaining: 6m 14s
17:	learn: 0.6869100	total: 6.88s	remaining: 6m 15s
18:	learn: 0.6864287	total: 7.38s	remaining: 6m 20s
19:	


The local accuracy is around 51. If we did not overfit, we shall expect something within the range above.

After submitting the benchmark file at https://challengedata.ens.fr, we obtain a public score of 51.31 %.